# Lesson 03: Exploring AI Agentic Frameworks

In this notebook, you will:
1. Build a modular `@tool` agent (`book_flight`) using `google-genai`.
2. Chain a `dataretrieval` agent and `dataanalysis` agent in sequence.
3. Implement a stateful `ThreadStore` + `Host` restaurant menu agent (`get_specials`, `get_item_price`).

In [ ]:
!pip install -q -U google-genai langgraph python-dotenv

In [ ]:
import os
from google import genai
from google.genai import types

def get_gemini_client():
    api_key = None
    try:
        from google.colab import userdata
        api_key = userdata.get("GEMINI_API_KEY")
    except Exception:
        pass
    api_key = api_key or os.environ.get("GEMINI_API_KEY")
    return genai.Client(api_key=api_key) if api_key else None

client = get_gemini_client()

def book_flight(date: str, location: str) -> str:
    """Book travel given location and date."""
    return f"Travel was booked to {location} on {date}."

if client:
    res = client.models.generate_content(
        model="gemini-2.5-flash",
        contents="Book a flight to New York on 2026-01-01.",
        config=types.GenerateContentConfig(tools=[book_flight])
    )
    print(res.text)
else:
    print("[Offline Simulation]", book_flight("2026-01-01", "New York"))

In [ ]:
# Stateful Thread Runtime Simulation
def get_specials() -> str:
    return "Special Soup: Clam Chowder | Special Salad: Cobb Salad | Special Drink: Chai Tea"

def get_item_price(menu_item: str) -> str:
    return "$9.99"

thread_messages = []
for user_input in ["What is the special soup?", "How much does that cost?"]:
    thread_messages.append({"role": "user", "text": user_input})
    reply = get_specials() if "soup" in user_input.lower() else get_item_price("Clam Chowder")
    thread_messages.append({"role": "assistant", "text": reply})
    print(f"User: {user_input}\nHost: {reply}\n")